In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from collections import defaultdict
from models.gbdt_uplift_model import TwoStageGradientBoostingUpliftClassifier
from utils.metrics import qini_score

DATASET_PATH = os.path.join(os.getcwd(), "data")


# --- HÀM CHẠY THỰC NGHIỆM ĐA NHIỆM (MULTI-TREATMENT) ---
def run_multitreatment_experiment(dataset_alias, model_params):
    print(f"Running Multi-Treatment Experiment on {dataset_alias}...")
    
    # Dictionary để lưu kết quả cho từng Treatment
    # Format: results[treat_id]['auuc'] = [fold1, fold2...]
    results = defaultdict(lambda: {'auuc': [], 'mse': []})
    
    for i in range(5):
        print(f"  > Processing Fold {i}...")
        
        # 1. Load Data
        folder = os.path.join(DATASET_PATH, f'{dataset_alias}_{i}')
        if not os.path.exists(folder):
            continue
        train = joblib.load(os.path.join(folder, 'train.pkl'))
        test = joblib.load(os.path.join(folder, 'test.pkl'))
        
        # 2. Train Model (JOINT TRAINING)
        # Đưa toàn bộ t (0, 1, 2...) vào. Model sẽ tự hiểu là Multi-class.
        model = TwoStageGradientBoostingUpliftClassifier(**model_params)
        model.fit(train['X'], train['y'], train['t'])
        
        # 3. Predict (Sẽ trả về Ma trận nếu có nhiều Treatment)
        # Output shape: (N_samples, N_treatments)
        uplift_preds = model.predict(test['X'])
        
        # 4. Evaluate từng Treatment
        n_treatments = train['t'].max() # Ví dụ: 2 treatment (1, 2)
        has_effect = 'effect' in test
        
        for t_id in range(1, n_treatments + 1):
            # a. Lấy cột dự đoán tương ứng (cột 0 là T1, cột 1 là T2...)
            if uplift_preds.ndim > 1:
                pred_t = uplift_preds[:, t_id - 1]
            else:
                pred_t = uplift_preds # Trường hợp chỉ có 1 treatment
            
            # b. Lọc dữ liệu đánh giá (Chỉ so sánh Treatment t_id vs Control)
            # Mask: Lấy những người thuộc nhóm Control (0) HOẶC nhóm Treatment đang xét (t_id)
            mask_eval = np.isin(test['t'], [0, t_id])
            
            y_eval = test['y'][mask_eval]
            # Chuyển về Binary: Treatment hiện tại thành 1, Control thành 0
            t_eval = (test['t'][mask_eval] == t_id).astype(int) 
            pred_eval = pred_t[mask_eval]
            
            # c. Tính AUUC
            _, score = qini_score(y_eval, pred_eval, t_eval)
            results[t_id]['auuc'].append(score)
            
            # d. Tính MSE (Nếu có effect)
            if has_effect:
                # Lấy effect thật của những người trong nhóm này
                eff_eval = test['effect'][mask_eval]
                mse = ((eff_eval - pred_eval)**2).mean()
                results[t_id]['mse'].append(mse)

    return results

def format_result(values):
    if len(values) == 0: return "N/A"
    return f"{np.mean(values):.4f} ± {np.std(values, ddof=1):.4f}"

if __name__ == '__main__':
    params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'n_estimators': 300,
        'uplift_ensemble_weight': 0.5,
        'verbose': False
    }
    
    # Chạy trên synth1 (Giả sử có nhiều treatment)
    # Hoặc đổi thành 'hillstrom' (nếu muốn test trên data thật)
    res_dict = run_multitreatment_experiment('synth1', params)
    
    # Tổng hợp ra bảng
    final_data = []
    for t_id in sorted(res_dict.keys()):
        row = {
            'Treatment': f'Treatment {t_id}',
            'AUUC': format_result(res_dict[t_id]['auuc']),
            'MSE': format_result(res_dict[t_id]['mse'])
        }
        final_data.append(row)
        
    df = pd.DataFrame(final_data)
    print("\n=== MULTI-TREATMENT RESULTS ===")
    print(df.to_string(index=False))